In [ ]:
from numba import cuda
import numpy as np
import math
import time

In [ ]:
@cuda.jit
def matrix_multiplication(A, B, C):

    row, col = cuda.grid(2)

    if row < C.shape[0] and col < C.shape[1]:
        temp = 0

        for k in range(A.shape[1]):
            temp += A[row][k] * B[k][col]

        C[row][col] = temp


In [ ]:
# Matrix Size

N = 26

# random matrix

A = np.random.randint(0,10,(N,N)).astype(np.float32)
B = np.random.randint(0,10,(N,N)).astype(np.float32)

# output martices
C_gpu = np.zeros((N,N)).astype(np.float32)


# copy date to gpu
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(C_gpu)

In [ ]:
# Threads Config

threads_per_block = (16,16)

blocks_per_grid_x = math.ceil(N / threads_per_block[0])
blocks_per_grid_y = math.ceil(N / threads_per_block[1])

blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

In [ ]:
# GPU Timining

start_gpu = time.perf_counter()

# launch kernel
matrix_multiplication[blocks_per_grid, threads_per_block](d_A,d_B,d_C)

cuda.synchronize()

C_gpu = d_C.copy_to_host()

end_gpu = time.perf_counter()

gpu_time = (end_gpu - start_gpu) * 1000

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [ ]:
# CPU Timining

start_cpu = time.perf_counter()

C_cpu = np.zeros((A.shape[0],B.shape[1]))

for i in range(A.shape[0]):
    for j in range(B.shape[0]):
        for k in range(A.shape[1]):
            C_cpu[i][j] += A[i][k] * B[k][j]

end_cpu = time.perf_counter()

cpu_time = (end_cpu - start_cpu) * 1000

In [ ]:
print("Matrix A:")
print(A[:5, :5])

print("\nMatrix B:")
print(B[:5, :5])

print("\nGPU Result:")
print(C_gpu[:5, :5])

print("\nCPU Result:")
print(C_cpu[:5, :5])

print("\nGPU Execution Time:", gpu_time, "ms")
print("CPU Execution Time:", cpu_time, "ms")

Matrix A:
[[2. 2. 6. 7. 9.]
 [6. 4. 9. 8. 2.]
 [4. 3. 1. 7. 2.]
 [9. 4. 4. 4. 3.]
 [1. 4. 6. 1. 0.]]

Matrix B:
[[5. 1. 2. 8. 7.]
 [7. 0. 6. 2. 9.]
 [8. 3. 1. 0. 8.]
 [9. 9. 8. 6. 9.]
 [8. 7. 6. 9. 9.]]

GPU Result:
[[613. 529. 530. 441. 590.]
 [702. 531. 615. 495. 651.]
 [752. 645. 732. 547. 694.]
 [568. 493. 585. 441. 552.]
 [612. 500. 507. 402. 553.]]

CPU Result:
[[613. 529. 530. 441. 590.]
 [702. 531. 615. 495. 651.]
 [752. 645. 732. 547. 694.]
 [568. 493. 585. 441. 552.]
 [612. 500. 507. 402. 553.]]

GPU Execution Time: 2.9474169999730293 ms
CPU Execution Time: 20.771418000094855 ms
